In [ ]:
pip install torch transformers datasets sentence-transformers accelerate

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from sentence_transformers import SentenceTransformer
import random
import re

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sentence_transformers import SentenceTransformer
import random
import re

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

dataset = load_dataset("gsm8k", "main", split="train[:300]")
dataset_test = load_dataset("gsm8k", "main", split="test[:100]")

model_name = "google/flan-t5-large"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)
model.eval()

#LLM call
def call_llm(question, prev_answer=None):
    if prev_answer:
        prompt = f"""
        Question: {question}
        Previous answer: {prev_answer}
        The previous solution may be incorrect.
        Re-solve carefully step by step and give the final answer after '####'.
        """
    else:
        prompt = f"""
        Question: {question}
        Let's think step by step.
        """

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        padding=True
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False,
            temperature=0.0
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response

#numerical extraction

def extract_gsm8k_answer(text):
    if "####" in text:
        return text.split("####")[-1].strip()
    nums = re.findall(r"-?\d+\.?\d*", text)
    return nums[-1] if nums else None

def extract_number(text):
    nums = re.findall(r"-?\d+\.?\d*", text)
    return nums[-1] if nums else None

#embed model

embed_model = SentenceTransformer("all-MiniLM-L6-v2")

#set environment
class SelfCorrectionEnv:
    def __init__(self, dataset, max_revisions=3):
        self.dataset = dataset
        self.max_revisions = max_revisions

    def reset(self):
        self.sample = random.choice(self.dataset)
        self.question = self.sample["question"]
        self.solution = self.sample["answer"]
        self.answer = None
        self.revisions = 0
        return self.get_state()

    def get_state(self):
        text = self.question
        if self.answer:
            text += " " + self.answer

        embedding = embed_model.encode(text)
        state = torch.tensor(embedding, dtype=torch.float32)
        revision_tensor = torch.tensor([self.revisions], dtype=torch.float32)

        return torch.cat([state, revision_tensor])

    def step(self, action):
        done = False

        # Action 0 = STOP
        if action == 0 or self.revisions >= self.max_revisions:
            done = True
            pred = extract_number(self.answer) if self.answer else None
            true = extract_number(self.solution)
            reward = 1 if pred == true else 0
            return self.get_state(), reward, done

        # Action 1 = REVISE
        self.answer = call_llm(self.question, self.answer)
        self.revisions += 1
        reward = -0.05  # small cost for revision

        return self.get_state(), reward, done

class PolicyNet(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 2),
            nn.Softmax(dim=-1)
        )

    def forward(self, x):
      probs = self.net(x)
      probs = torch.clamp(probs, min=1e-8, max=1.0)
      probs = probs / probs.sum()
      return probs

def train(env, policy, episodes=200):
    optimizer = optim.Adam(policy.parameters(), lr=5e-5)
    gamma = 0.99

    for ep in range(episodes):
        state = env.reset()
        log_probs = []
        rewards = []
        done = False

        while not done:
            state = state.to(device)
            probs = policy(state)
            dist = torch.distributions.Categorical(probs)
            action = dist.sample()

            log_probs.append(dist.log_prob(action))
            next_state, reward, done = env.step(action.item())
            rewards.append(reward)
            state = next_state

        # Discount rewards
        G = 0
        returns = []
        for r in reversed(rewards):
            G = r + gamma * G
            returns.insert(0, G)

        returns = torch.tensor(returns).to(device)

        if len(returns) > 1:
          returns = (returns - returns.mean()) / (returns.std() + 1e-8)

        loss = 0
        for log_prob, R in zip(log_probs, returns):
            loss -= log_prob * R

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if ep % 10 == 0:
            print(f"Episode {ep} | Total Reward: {sum(rewards):.2f}")

env = SelfCorrectionEnv(dataset)
input_dim = env.reset().shape[0]

policy = PolicyNet(input_dim).to(device)

train(env, policy, episodes=100)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Episode 0 | Total Reward: -0.10
Episode 10 | Total Reward: 0.00
Episode 20 | Total Reward: 0.00
Episode 30 | Total Reward: -0.10
Episode 40 | Total Reward: 0.00
Episode 50 | Total Reward: -0.05
Episode 60 | Total Reward: 0.00
Episode 70 | Total Reward: 0.00
Episode 80 | Total Reward: -0.05
Episode 90 | Total Reward: -0.15


using stronger model-- flan-t5-larger

In [ ]:
print(model.config._name_or_path)

google/flan-t5-base


In [ ]:
sample = dataset[0]

print("QUESTION:")
print(sample["question"])

response = call_llm(sample["question"])

print("\nMODEL OUTPUT:")
print(response)

print("\nTRUE ANSWER:")
print(sample["answer"])

QUESTION:
Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?

MODEL OUTPUT:
In April, Natalia sold 48 / 2 = 24 clips. In May, Natalia sold 48 / 2 = 18 clips. In April and May, Natalia sold 48 + 18 = 96 clips. The answer: 96.

TRUE ANSWER:
Natalia sold 48/2 = <<48/2=24>>24 clips in May.
Natalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.
#### 72


is not final

In [ ]:
# Cell 1 — download raw files with wget (bypasses HF client timeout)
!wget -q "https://huggingface.co/datasets/openai/gsm8k/resolve/main/main/train-00000-of-00001.parquet" -O gsm8k_train.parquet
!wget -q "https://huggingface.co/datasets/openai/gsm8k/resolve/main/main/test-00000-of-00001.parquet"  -O gsm8k_test.parquet

In [ ]:
# Cell 2 — load from local files
import pandas as pd
from datasets import Dataset

df_train = pd.read_parquet("gsm8k_train.parquet").head(300)
df_test  = pd.read_parquet("gsm8k_test.parquet").head(100)

dataset      = Dataset.from_pandas(df_train)
dataset_test = Dataset.from_pandas(df_test)

print(dataset[0])   # verify it loaded

{'question': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?', 'answer': 'Natalia sold 48/2 = <<48/2=24>>24 clips in May.\nNatalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.\n#### 72'}


In [ ]:
"""
RL-based Self-Correction for Math Reasoning (GSM8K)
====================================================
Fixes applied vs original:
  1. Initial LLM call in env.reset() so answer is never None
  2. Shaped rewards (not sparse +1/0)
  3. extract_gsm8k_answer used consistently for ground truth
  4. Correct model verified after loading
  5. Gradient clipping added for stability
  6. Baseline subtracted per-episode (reduces variance)
  7. Evaluation loop added after training
"""

import torch
import torch.nn as nn
import torch.optim as optim
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sentence_transformers import SentenceTransformer
import random
import re

# ── Device ────────────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ── Data ──────────────────────────────────────────────────────────────────────
dataset      = load_dataset("gsm8k", "main", split="train[:300]")
dataset_test = load_dataset("gsm8k", "main", split="test[:100]")

# ── LLM ───────────────────────────────────────────────────────────────────────
# NOTE: flan-t5-large ~780 M params; upgrade to flan-t5-xl/xxl or a 7 B model
#       (Mistral-7B, Llama-3-8B) for meaningfully better GSM8K accuracy.
model_name = "google/flan-t5-large"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model     = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)
model.eval()

# Sanity-check: confirm the right checkpoint actually loaded
print(f"Loaded model: {model.config._name_or_path}")
assert "large" in model.config._name_or_path, (
    "Wrong checkpoint loaded — clear HF cache and retry."
)


# ── LLM call ─────────────────────────────────────────────────────────────────
def call_llm(question: str, prev_answer: str | None = None) -> str:
    if prev_answer:
        prompt = (
            f"Question: {question}\n"
            f"Previous answer: {prev_answer}\n"
            "The previous solution may be incorrect. "
            "Re-solve carefully step by step and give the final answer after '####'."
        )
    else:
        prompt = (
            f"Question: {question}\n"
            "Let's think step by step."
        )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512,
        padding=True,
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False,
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)


# ── Answer extraction ─────────────────────────────────────────────────────────
def extract_gsm8k_answer(text: str) -> str | None:
    """Extract ground-truth answer from GSM8K '#### <num>' format."""
    if "####" in text:
        return text.split("####")[-1].strip().replace(",", "")
    nums = re.findall(r"-?\d+\.?\d*", text)
    return nums[-1] if nums else None


def extract_number(text: str) -> str | None:
    """Extract the last number from free-form model output."""
    if text is None:
        return None
    # Prefer #### format first
    if "####" in text:
        return text.split("####")[-1].strip().replace(",", "")
    nums = re.findall(r"-?\d[\d,]*\.?\d*", text)
    if not nums:
        return None
    return nums[-1].replace(",", "")


def answers_match(pred: str | None, true: str | None) -> bool:
    if pred is None or true is None:
        return False
    try:
        return abs(float(pred) - float(true)) < 1e-3
    except ValueError:
        return pred.strip() == true.strip()


# ── Sentence embedding model ──────────────────────────────────────────────────
embed_model = SentenceTransformer("all-MiniLM-L6-v2")
EMBED_DIM   = embed_model.get_sentence_embedding_dimension()  # 384


# ── Environment ───────────────────────────────────────────────────────────────
class SelfCorrectionEnv:
    """
    State  : sentence embedding of (question + current answer) + revision count
    Actions: 0 = STOP  |  1 = REVISE

    Reward shaping
    ──────────────
    STOP   correct              → +1.0
    STOP   incorrect            → -0.5   (penalise giving up on a wrong answer)
    REVISE correct → correct    → -0.05  (wasted revision)
    REVISE wrong   → correct    → +0.8   (good self-correction)
    REVISE correct → wrong      → -0.3   (harmful revision)
    REVISE wrong   → wrong      → -0.05  (unhelpful but not catastrophic)
    """

    def __init__(self, dataset, max_revisions: int = 3):
        self.dataset       = dataset
        self.max_revisions = max_revisions

    # ------------------------------------------------------------------
    def reset(self):
        self.sample   = random.choice(self.dataset)
        self.question = self.sample["question"]
        self.solution = self.sample["answer"]          # raw GSM8K answer text
        self.true_ans = extract_gsm8k_answer(self.solution)
        self.revisions = 0

        # FIX 1: always get an initial answer so state is never None
        self.answer = call_llm(self.question)

        return self._get_state()

    # ------------------------------------------------------------------
    def _get_state(self) -> torch.Tensor:
        text      = self.question + " " + (self.answer or "")
        embedding = embed_model.encode(text)
        state     = torch.tensor(embedding, dtype=torch.float32)
        rev_feat  = torch.tensor([self.revisions / self.max_revisions],
                                 dtype=torch.float32)   # normalised
        return torch.cat([state, rev_feat])

    # ------------------------------------------------------------------
    def step(self, action: int):
        done = False

        # ── STOP ──────────────────────────────────────────────────────
        if action == 0 or self.revisions >= self.max_revisions:
            done      = True
            pred      = extract_number(self.answer)
            # FIX 2: shaped reward — penalise wrong stops too
            correct   = answers_match(pred, self.true_ans)
            reward    = 1.0 if correct else -0.5
            return self._get_state(), reward, done

        # ── REVISE ────────────────────────────────────────────────────
        prev_correct = answers_match(extract_number(self.answer), self.true_ans)

        self.answer    = call_llm(self.question, self.answer)
        self.revisions += 1

        new_correct = answers_match(extract_number(self.answer), self.true_ans)

        # Shaped revision rewards
        if new_correct and not prev_correct:
            reward = +0.8   # successfully self-corrected
        elif prev_correct and not new_correct:
            reward = -0.3   # revision broke a correct answer
        else:
            reward = -0.05  # neutral (wrong→wrong or correct→correct)

        return self._get_state(), reward, done


# ── Policy network ────────────────────────────────────────────────────────────
class PolicyNet(nn.Module):
    def __init__(self, input_dim: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 2),
            nn.Softmax(dim=-1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        probs = self.net(x)
        probs = torch.clamp(probs, min=1e-8)
        return probs / probs.sum()


# ── REINFORCE training ────────────────────────────────────────────────────────
def train(env: SelfCorrectionEnv, policy: PolicyNet, episodes: int = 200):
    optimizer   = optim.Adam(policy.parameters(), lr=1e-4)
    gamma       = 0.99
    reward_log  = []

    for ep in range(episodes):
        state     = env.reset()
        log_probs = []
        rewards   = []
        done      = False

        while not done:
            state = state.to(device)
            probs = policy(state)
            dist  = torch.distributions.Categorical(probs)
            action = dist.sample()

            log_probs.append(dist.log_prob(action))
            next_state, reward, done = env.step(action.item())
            rewards.append(reward)
            state = next_state

        # ── Discounted returns ────────────────────────────────────────
        G       = 0.0
        returns = []
        for r in reversed(rewards):
            G = r + gamma * G
            returns.insert(0, G)

        returns = torch.tensor(returns, dtype=torch.float32).to(device)

        # FIX 3: subtract mean as baseline (reduces variance even for len=1)
        returns = returns - returns.mean()
        if len(returns) > 1:
            returns = returns / (returns.std() + 1e-8)

        # ── Policy gradient loss ──────────────────────────────────────
        loss = torch.stack([-lp * R for lp, R in zip(log_probs, returns)]).sum()

        optimizer.zero_grad()
        loss.backward()
        # FIX 4: gradient clipping for stability
        nn.utils.clip_grad_norm_(policy.parameters(), max_norm=1.0)
        optimizer.step()

        ep_reward = sum(rewards)
        reward_log.append(ep_reward)

        if ep % 10 == 0:
            avg = sum(reward_log[-10:]) / min(10, len(reward_log))
            print(f"Episode {ep:4d} | Reward: {ep_reward:+.2f} | "
                  f"10-ep avg: {avg:+.2f} | Revisions: {env.revisions}")

    return reward_log


# ── Evaluation ────────────────────────────────────────────────────────────────
@torch.no_grad()
def evaluate(dataset, policy: PolicyNet, n: int = 50) -> dict:
    """
    Compare three strategies on n random test problems:
      - zero_shot   : single LLM call, no revision
      - always_revise: always revise max_revisions times
      - rl_policy   : policy decides when to stop
    """
    policy.eval()
    env = SelfCorrectionEnv(dataset, max_revisions=3)

    results = {"zero_shot": 0, "always_revise": 0, "rl_policy": 0}
    samples = random.sample(list(dataset), n)

    for sample in samples:
        question  = sample["question"]
        solution  = sample["answer"]
        true_ans  = extract_gsm8k_answer(solution)

        # ── zero-shot ─────────────────────────────────────────────────
        ans0 = call_llm(question)
        if answers_match(extract_number(ans0), true_ans):
            results["zero_shot"] += 1

        # ── always revise ─────────────────────────────────────────────
        ans = ans0
        for _ in range(3):
            ans = call_llm(question, ans)
        if answers_match(extract_number(ans), true_ans):
            results["always_revise"] += 1

        # ── RL policy ─────────────────────────────────────────────────
        # Manually step through env with policy control
        env.question   = question
        env.solution   = solution
        env.true_ans   = true_ans
        env.answer     = ans0
        env.revisions  = 0
        state          = env._get_state()
        done           = False

        while not done:
            probs  = policy(state.to(device))
            action = probs.argmax().item()           # greedy at eval time
            state, _, done = env.step(action)

        if answers_match(extract_number(env.answer), true_ans):
            results["rl_policy"] += 1

    pct = {k: v / n * 100 for k, v in results.items()}
    print("\n── Evaluation Results ──────────────────────────────")
    for k, v in pct.items():
        print(f"  {k:<20s}: {v:.1f}%  ({results[k]}/{n})")
    return pct


# ── Main ──────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    env       = SelfCorrectionEnv(dataset, max_revisions=3)
    input_dim = env.reset().shape[0]   # EMBED_DIM + 1
    print(f"State dim: {input_dim}")

    policy = PolicyNet(input_dim).to(device)

    print("\n── Training ────────────────────────────────────────")
    reward_log = train(env, policy, episodes=200)

    print("\n── Saving policy ───────────────────────────────────")
    torch.save(policy.state_dict(), "policy.pt")
    print("Saved to policy.pt")

    print("\n── Evaluating ──────────────────────────────────────")
    evaluate(dataset_test, policy, n=50)

Using device: cpu


main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

/tmp/ipykernel_14414/3655147324.py:113: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  EMBED_DIM   = embed_model.get_sentence_embedding_dimension()  # 384


State dim: 385

── Training ────────────────────────────────────────
Episode    0 | Reward: -0.55 | 10-ep avg: -0.55 | Revisions: 1
Episode   10 | Reward: -0.50 | 10-ep avg: -0.24 | Revisions: 0
Episode   20 | Reward: -0.50 | 10-ep avg: -0.24 | Revisions: 0
Episode   30 | Reward: -0.65 | 10-ep avg: -0.44 | Revisions: 3
Episode   40 | Reward: +1.00 | 10-ep avg: -0.21 | Revisions: 0
Episode   50 | Reward: -0.60 | 10-ep avg: -0.39 | Revisions: 2
Episode   60 | Reward: -0.50 | 10-ep avg: -0.55 | Revisions: 0
Episode   70 | Reward: +1.00 | 10-ep avg: -0.24 | Revisions: 0
Episode   80 | Reward: -0.60 | 10-ep avg: -0.53 | Revisions: 2
Episode   90 | Reward: -0.65 | 10-ep avg: -0.39 | Revisions: 3
Episode  100 | Reward: -0.50 | 10-ep avg: -0.36 | Revisions: 0
Episode  110 | Reward: -0.55 | 10-ep avg: -0.53 | Revisions: 1
Episode  120 | Reward: -0.55 | 10-ep avg: -0.38 | Revisions: 1
Episode  130 | Reward: -0.55 | 10-ep avg: -0.10 | Revisions: 1
Episode  140 | Reward: -0.50 | 10-ep avg: -0.40 |

chance final

In [ ]:
!pip install sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.3/571.3 kB 21.7 MB/s eta 0:00:00


In [ ]:
"""
RL-based Self-Correction for Math Reasoning (GSM8K)
====================================================
Changes vs previous version:
  1.  Stronger model: Mistral-7B-Instruct (via HF) instead of flan-t5-large
      → flan-t5-large scores ~6% on GSM8K; a 7B model scores 35-50%,
        giving the policy real signal to learn from.
      Fallback: set USE_SMALL_MODEL=True to keep flan-t5-large for CPU testing.
  2.  Running-average baseline instead of per-episode mean subtraction
      → per-episode mean zeroes out 1-step returns entirely (zero gradient).
  3.  Entropy bonus in policy loss to prevent premature action collapse.
  4.  Entropy + loss logging every 10 episodes for diagnostics.
  5.  Clean evaluation loop that never mutates env internals.
  6.  4-bit quantization (bitsandbytes) enabled when CUDA is available so
      Mistral-7B fits in ~6 GB VRAM.
  7.  Minor: removed broken `if len(returns) > 1` std guard — now always
      normalises after running-baseline subtraction when episode length > 1.
"""

import os
import torch
import torch.nn as nn
import torch.optim as optim
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSeq2SeqLM
from sentence_transformers import SentenceTransformer
import random
import re

# ── Config ────────────────────────────────────────────────────────────────────
# Set USE_SMALL_MODEL=True if you only have CPU / want a quick smoke-test.
# The small model (flan-t5-large) tops out at ~6% GSM8K accuracy, so the RL
# policy cannot meaningfully improve over baselines in that setting.
USE_SMALL_MODEL = os.environ.get("USE_SMALL_MODEL", "false").lower() == "true"

ENTROPY_COEF  = 0.05    # weight for entropy bonus in policy loss
BASELINE_ALPHA = 0.05   # EMA smoothing for running reward baseline
LR            = 1e-4
GAMMA         = 0.99
MAX_REVISIONS = 3
TRAIN_EPISODES = 500
EVAL_N         = 50

# ── Device ────────────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ── Data ──────────────────────────────────────────────────────────────────────
dataset      = load_dataset("gsm8k", "main", split="train[:300]")
dataset_test = load_dataset("gsm8k", "main", split="test[:100]")

# ── LLM ───────────────────────────────────────────────────────────────────────
if USE_SMALL_MODEL:
    # ── flan-t5-large (Seq2Seq, ~780M) — CPU-friendly smoke-test only ─────────
    model_name  = "google/flan-t5-large"
    tokenizer   = AutoTokenizer.from_pretrained(model_name)
    llm         = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)
    IS_CAUSAL   = False
    print(f"[WARNING] Using flan-t5-large. GSM8K accuracy is ~6%, "
          "so RL policy cannot improve over baselines. "
          "Set USE_SMALL_MODEL=false and use a GPU for real experiments.")
else:
    # ── Mistral-7B-Instruct (CausalLM) — recommended ─────────────────────────
    # 4-bit quantization requires:  pip install bitsandbytes>=0.41.0
    # Falls back to fp16 if bitsandbytes is unavailable.
    model_name = "mistralai/Mistral-7B-Instruct-v0.2"
    tokenizer  = AutoTokenizer.from_pretrained(model_name)
    tokenizer.pad_token = tokenizer.eos_token

    try:
        from transformers import BitsAndBytesConfig
        bnb_cfg = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        )
        llm = AutoModelForCausalLM.from_pretrained(
            model_name,
            quantization_config=bnb_cfg,
            device_map="auto",
        )
        print("Loaded Mistral-7B in 4-bit quantization.")
    except (ImportError, Exception) as e:
        print(f"bitsandbytes not available ({e}); loading in fp16.")
        llm = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=torch.float16,
            device_map="auto",
        )
    IS_CAUSAL = True

llm.eval()
print(f"Loaded model: {llm.config._name_or_path}")


# ── LLM call ─────────────────────────────────────────────────────────────────
def call_llm(question: str, prev_answer: str | None = None) -> str:
    if prev_answer:
        if IS_CAUSAL:
            prompt = (
                f"[INST] Question: {question}\n\n"
                f"Previous answer: {prev_answer}\n\n"
                "The previous solution may contain an error. "
                "Re-solve carefully step by step. "
                "End your answer with '#### <number>'.[/INST]"
            )
        else:
            prompt = (
                f"Question: {question}\n"
                f"Previous answer: {prev_answer}\n"
                "The previous solution may be incorrect. "
                "Re-solve carefully step by step and give the final answer after '####'."
            )
    else:
        if IS_CAUSAL:
            prompt = (
                f"[INST] Solve this math problem step by step. "
                f"End your answer with '#### <number>'.\n\n"
                f"Question: {question}[/INST]"
            )
        else:
            prompt = (
                f"Question: {question}\n"
                "Let's think step by step."
            )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512,
        padding=True,
    ).to(device)

    with torch.no_grad():
        if IS_CAUSAL:
            input_len = inputs["input_ids"].shape[1]
            outputs   = llm.generate(
                **inputs,
                max_new_tokens=256,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
            # Decode only the newly generated tokens
            generated = outputs[0][input_len:]
        else:
            outputs   = llm.generate(**inputs, max_new_tokens=200, do_sample=False)
            generated = outputs[0]

    return tokenizer.decode(generated, skip_special_tokens=True)


# ── Answer extraction ─────────────────────────────────────────────────────────
def extract_gsm8k_answer(text: str) -> str | None:
    """Extract ground-truth answer from GSM8K '#### <num>' format."""
    if "####" in text:
        return text.split("####")[-1].strip().replace(",", "")
    nums = re.findall(r"-?\d+\.?\d*", text)
    return nums[-1] if nums else None


def extract_number(text: str) -> str | None:
    """Extract the last number from free-form model output."""
    if text is None:
        return None
    if "####" in text:
        return text.split("####")[-1].strip().replace(",", "")
    nums = re.findall(r"-?\d[\d,]*\.?\d*", text)
    return nums[-1].replace(",", "") if nums else None


def answers_match(pred: str | None, true: str | None) -> bool:
    if pred is None or true is None:
        return False
    try:
        return abs(float(pred) - float(true)) < 1e-3
    except ValueError:
        return pred.strip() == true.strip()


# ── Sentence embedding model ──────────────────────────────────────────────────
embed_model = SentenceTransformer("all-MiniLM-L6-v2")
EMBED_DIM   = embed_model.get_sentence_embedding_dimension()  # 384


# ── Environment ───────────────────────────────────────────────────────────────
class SelfCorrectionEnv:
    """
    State  : sentence embedding of (question + current answer) + revision count
    Actions: 0 = STOP  |  1 = REVISE

    Reward shaping
    ──────────────
    STOP   correct              → +1.0
    STOP   incorrect            → -0.5   (penalise giving up on wrong answer)
    REVISE wrong   → correct    → +0.8   (good self-correction)
    REVISE correct → wrong      → -0.3   (harmful revision)
    REVISE correct → correct    → -0.05  (wasted revision)
    REVISE wrong   → wrong      → -0.05  (unhelpful but not catastrophic)
    """

    def __init__(self, dataset, max_revisions: int = MAX_REVISIONS):
        self.dataset       = dataset
        self.max_revisions = max_revisions

    def reset(self):
        self.sample    = random.choice(self.dataset)
        self.question  = self.sample["question"]
        self.solution  = self.sample["answer"]
        self.true_ans  = extract_gsm8k_answer(self.solution)
        self.revisions = 0
        self.answer    = call_llm(self.question)   # always initialise
        return self._get_state()

    def _get_state(self) -> torch.Tensor:
        text      = self.question + " " + (self.answer or "")
        embedding = embed_model.encode(text)
        state     = torch.tensor(embedding, dtype=torch.float32)
        rev_feat  = torch.tensor(
            [self.revisions / self.max_revisions], dtype=torch.float32
        )
        return torch.cat([state, rev_feat])

    def step(self, action: int):
        done = False

        if action == 0 or self.revisions >= self.max_revisions:
            done    = True
            pred    = extract_number(self.answer)
            correct = answers_match(pred, self.true_ans)
            reward  = 1.0 if correct else -0.5
            return self._get_state(), reward, done

        prev_correct = answers_match(extract_number(self.answer), self.true_ans)
        self.answer   = call_llm(self.question, self.answer)
        self.revisions += 1
        new_correct  = answers_match(extract_number(self.answer), self.true_ans)

        if new_correct and not prev_correct:
            reward = +0.8
        elif prev_correct and not new_correct:
            reward = -0.3
        else:
            reward = -0.05

        return self._get_state(), reward, done


# ── Policy network ────────────────────────────────────────────────────────────
class PolicyNet(nn.Module):
    def __init__(self, input_dim: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 2),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Return raw logits; Categorical handles the softmax internally
        return self.net(x)


# ── REINFORCE training ────────────────────────────────────────────────────────
def train(env: SelfCorrectionEnv, policy: PolicyNet, episodes: int = TRAIN_EPISODES):
    optimizer    = optim.Adam(policy.parameters(), lr=LR)
    reward_log   = []
    running_baseline = 0.0   # FIX: EMA baseline across episodes, not within-episode mean

    for ep in range(episodes):
        policy.train()
        state     = env.reset()
        log_probs = []
        entropies = []
        rewards   = []
        done      = False

        while not done:
            state  = state.to(device)
            logits = policy(state)
            dist   = torch.distributions.Categorical(logits=logits)
            action = dist.sample()

            log_probs.append(dist.log_prob(action))
            entropies.append(dist.entropy())

            next_state, reward, done = env.step(action.item())
            rewards.append(reward)
            state = next_state

        # ── Discounted returns ────────────────────────────────────────
        G       = 0.0
        returns = []
        for r in reversed(rewards):
            G = r + GAMMA * G
            returns.insert(0, G)

        returns = torch.tensor(returns, dtype=torch.float32).to(device)

        # FIX: subtract running EMA baseline (works correctly for 1-step episodes)
        running_baseline = (
            (1 - BASELINE_ALPHA) * running_baseline
            + BASELINE_ALPHA * returns.mean().item()
        )
        returns = returns - running_baseline

        # Normalise std only when episode has >1 step (avoid div-by-zero on scalar)
        if len(returns) > 1:
            returns = returns / (returns.std() + 1e-8)

        # ── Policy gradient loss + entropy bonus ──────────────────────
        # FIX: entropy bonus prevents the policy collapsing to always-STOP
        pg_loss      = torch.stack([-lp * R for lp, R in zip(log_probs, returns)]).sum()
        entropy_loss = torch.stack(entropies).sum()
        loss         = pg_loss - ENTROPY_COEF * entropy_loss

        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(policy.parameters(), max_norm=1.0)
        optimizer.step()

        ep_reward = sum(rewards)
        reward_log.append(ep_reward)

        if ep % 10 == 0:
            avg     = sum(reward_log[-10:]) / min(10, len(reward_log))
            avg_ent = entropy_loss.item() / len(entropies)
            print(
                f"Episode {ep:4d} | Reward: {ep_reward:+.2f} | "
                f"10-ep avg: {avg:+.2f} | Revisions: {env.revisions} | "
                f"Loss: {loss.item():+.4f} | Entropy: {avg_ent:.3f}"
            )

    return reward_log


# ── Evaluation ────────────────────────────────────────────────────────────────
@torch.no_grad()
def evaluate(dataset, policy: PolicyNet, n: int = EVAL_N) -> dict:
    """
    Compare three strategies on n random test problems:
      - zero_shot    : single LLM call, no revision
      - always_revise: always revise MAX_REVISIONS times
      - rl_policy    : policy decides when to stop (greedy)
    """
    policy.eval()
    results = {"zero_shot": 0, "always_revise": 0, "rl_policy": 0}
    samples = random.sample(list(dataset), n)

    for i, sample in enumerate(samples):
        question = sample["question"]
        true_ans = extract_gsm8k_answer(sample["answer"])

        # ── zero-shot ─────────────────────────────────────────────────
        ans0 = call_llm(question)
        if answers_match(extract_number(ans0), true_ans):
            results["zero_shot"] += 1

        # ── always revise ─────────────────────────────────────────────
        ans = ans0
        for _ in range(MAX_REVISIONS):
            ans = call_llm(question, ans)
        if answers_match(extract_number(ans), true_ans):
            results["always_revise"] += 1

        # ── RL policy (clean loop — never mutates env internals) ──────
        # FIX: build state directly without touching env private state
        current_answer = ans0
        revisions      = 0
        done           = False

        while not done:
            text      = question + " " + current_answer
            embedding = embed_model.encode(text)
            state     = torch.tensor(embedding, dtype=torch.float32)
            rev_feat  = torch.tensor(
                [revisions / MAX_REVISIONS], dtype=torch.float32
            )
            state = torch.cat([state, rev_feat]).to(device)

            logits = policy(state)
            action = logits.argmax().item()   # greedy at eval time

            if action == 0 or revisions >= MAX_REVISIONS:
                done = True
            else:
                current_answer = call_llm(question, current_answer)
                revisions += 1

        if answers_match(extract_number(current_answer), true_ans):
            results["rl_policy"] += 1

        if (i + 1) % 10 == 0:
            print(f"  Evaluated {i+1}/{n} samples...")

    pct = {k: v / n * 100 for k, v in results.items()}
    print("\n── Evaluation Results ──────────────────────────────")
    for k, v in pct.items():
        print(f"  {k:<20s}: {v:.1f}%  ({results[k]}/{n})")
    return pct


# ── Main ──────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    env       = SelfCorrectionEnv(dataset, max_revisions=MAX_REVISIONS)
    input_dim = env.reset().shape[0]   # EMBED_DIM + 1 = 385
    print(f"State dim: {input_dim}")

    policy = PolicyNet(input_dim).to(device)

    print("\n── Training ────────────────────────────────────────")
    reward_log = train(env, policy, episodes=TRAIN_EPISODES)

    print("\n── Saving policy ───────────────────────────────────")
    torch.save(policy.state_dict(), "policy.pt")
    print("Saved to policy.pt")

    print("\n── Evaluating ──────────────────────────────────────")
    evaluate(dataset_test, policy, n=EVAL_N)

Using device: cpu


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

/tmp/ipykernel_725/1532375490.py:183: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  EMBED_DIM   = embed_model.get_sentence_embedding_dimension()  # 384


State dim: 385

── Training ────────────────────────────────────────
Episode    0 | Reward: -0.50 | 10-ep avg: -0.50 | Revisions: 0 | Loss: -0.3696 | Entropy: 0.693
Episode   10 | Reward: -0.55 | 10-ep avg: -0.38 | Revisions: 1 | Loss: -16.0054 | Entropy: 0.693


idddkkk

In [ ]:
pip install transformers datasets trl accelerate peft

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 697.4/697.4 kB 41.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 19.2 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
